# Step 1: Create Workspaces & Add Users

This notebook creates Fabric workspaces for each team and adds individual users as **Contributors**.

## Prerequisites

1. You must have a **Lakehouse** in this workspace
2. Upload `teams-template.xlsx` to the Lakehouse (or update it directly)
3. Fill in real team names and email addresses
4. You must have **Fabric workspace creation permissions** and **capacity assignment permissions**

## Configuration

Edit the values below before running:

In [ ]:
# Configuration — edit these
EXCEL_PATH = "Files/teams-template.xlsx"  # Path to teams file in Lakehouse
WORKSPACE_PREFIX = "bis-day"  # Prefix for workspace names: bis-day-team01, bis-day-team02, etc.
CAPACITY_ID = "21F29096-B423-437C-AB24-60F1830444A2"  # Your Fabric capacity ID — UPDATE THIS!
DRY_RUN = True  # Set to False to actually create workspaces

## Step 1: Load Team Data from Excel

In [ ]:
import pandas as pd

print(f"📂 Loading Excel from Lakehouse: {EXCEL_PATH}")
df = pd.read_excel(f"/lakehouse/default/{EXCEL_PATH}")

# Validate columns
required_columns = {"TeamName", "MemberEmail"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(
        f"❌ Missing required columns: {missing}\n"
        f"   Found: {df.columns.tolist()}"
    )

print(f"✅ Loaded {len(df)} team members")
print(f"\nTeams found:")
for team in df["TeamName"].unique():
    members = len(df[df["TeamName"] == team])
    print(f"  - {team}: {members} members")

## Step 2: Get Fabric API Token

In [ ]:
import notebookutils

print("🔐 Getting Fabric API token...")
fabric_token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
fabric_headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json",
}
print("✅ Token acquired")

## Step 3: Create Workspaces and Add Users

In [ ]:
import requests
import json

if DRY_RUN:
    print("🔍 DRY RUN MODE - No changes will be made\n")

results = []

def create_workspace(workspace_name: str) -> str | None:
    """Create a Fabric workspace. Returns workspace ID."""
    if DRY_RUN:
        print(f"  [DRY RUN] Would create workspace: {workspace_name}")
        return "00000000-0000-0000-0000-000000000000"

    payload = {"displayName": workspace_name}
    response = requests.post(
        "https://api.fabric.microsoft.com/v1/workspaces",
        headers=fabric_headers,
        json=payload,
    )

    if response.status_code == 201:
        workspace_id = response.json().get("id")
        print(f"  ✅ Created workspace: {workspace_name}")
        return workspace_id
    elif response.status_code == 400:
        # Workspace exists - find it
        print(f"  ⚠️  Workspace exists: {workspace_name}. Finding ID...")
        existing_id = find_workspace_by_name(workspace_name)
        if existing_id:
            print(f"  ✅ Using existing workspace")
            return existing_id
    else:
        print(f"  ❌ Failed to create workspace: {response.status_code}")
        print(f"     {response.text}")
    return None

def find_workspace_by_name(workspace_name: str) -> str | None:
    """Find workspace ID by display name."""
    response = requests.get(
        "https://api.fabric.microsoft.com/v1/workspaces",
        headers=fabric_headers,
    )
    if response.status_code == 200:
        for workspace in response.json().get("value", []):
            if workspace.get("displayName") == workspace_name:
                return workspace.get("id")
    return None

def assign_capacity(workspace_id: str, capacity_id: str) -> bool:
    """Assign capacity to workspace."""
    if DRY_RUN:
        print(f"    [DRY RUN] Would assign capacity")
        return True

    payload = {"capacityId": capacity_id}
    response = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/assignToCapacity",
        headers=fabric_headers,
        json=payload,
    )
    if response.status_code in (200, 202):
        print(f"    ✓ Assigned capacity")
        return True
    else:
        print(f"    ✗ Failed to assign capacity: {response.status_code}")
        print(f"      {response.text[:200]}")
    return False

def add_user_to_workspace(workspace_id: str, user_email: str, role: str = "Contributor") -> bool:
    """Add user to workspace using Power BI API (accepts email addresses)."""
    if DRY_RUN:
        print(f"      [DRY RUN] Would add {user_email} as {role}")
        return True

    # Use Power BI API which accepts email addresses directly
    pbi_headers = {
        "Authorization": fabric_headers["Authorization"],
        "Content-Type": "application/json",
    }
    payload = {
        "emailAddress": user_email,
        "groupUserAccessRight": role,
    }
    response = requests.post(
        f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/users",
        headers=pbi_headers,
        json=payload,
    )
    if response.status_code == 200:
        print(f"      ✓ Added {user_email} as {role}")
        return True
    else:
        print(f"      ✗ Failed: {response.status_code} - {response.text[:200]}")
    return False

# Process each team
teams = df["TeamName"].unique()
print(f"\n🚀 Processing {len(teams)} teams...\n")

for team in teams:
    workspace_name = f"{WORKSPACE_PREFIX}-{team}"
    print(f"📦 {team}")

    # Create workspace
    workspace_id = create_workspace(workspace_name)
    if not workspace_id:
        print(f"  ❌ Failed to create workspace")
        continue

    # Assign capacity
    if not DRY_RUN:
        assign_capacity(workspace_id, CAPACITY_ID)

    # Add team members (deduplicated)
    team_members = df[df["TeamName"] == team]["MemberEmail"].unique().tolist()
    for member_email in team_members:
        add_user_to_workspace(workspace_id, member_email, role="Contributor")
        results.append({
            "TeamName": team,
            "WorkspaceName": workspace_name,
            "WorkspaceId": workspace_id,
            "MemberEmail": member_email,
            "Role": "Contributor",
        })

    print()

print(f"✨ Done! Processed {len(results)} team member assignments.")

## Step 4: Save Results to Lakehouse

In [ ]:
if not DRY_RUN and results:
    print("💾 Saving results to Lakehouse...")
    results_df = pd.DataFrame(results)
    
    # Save to Lakehouse
    output_path = "/lakehouse/default/Files/teams-resolved.xlsx"
    results_df.to_excel(output_path, index=False, sheet_name="Teams")
    print(f"✅ Saved {len(results)} records to teams-resolved.xlsx")
    print(f"\n📊 Summary:")
    display(results_df.head(20))
else:
    print("⏭️  Skipping save (dry run mode)")